In [37]:
import pandas as pd

In [38]:
data = pd.read_csv('data/ingredients.csv', sep=',')
display(data)

,id,name,packing,store,price,type
0,1,сахар,1 кг,лента,62.99,продукты
1,2,печенье на декор,1 упаковка,лента,89.99,декор
2,3,сгущенка рогачев,370 мл,лента,164.99,продукты
3,4,малина и голубика в сердце,1 упаковка,перекресток,579.99,декор
4,5,яйцо с1 окское,10 шт,лента,109.99,продукты
...,...,...,...,...,...,...
133,134,глазурь темная,1 кг,ольга,560.00,декор
134,135,трайфлы,100 шт,ольга,1200.00,упаковка
135,136,коробка кап6,1 шт,ольга,60.00,упаковка
136,137,яйцо с0 365 дней,10 штук,лента,99.99,продукты


In [39]:
data[data['type'] =='продукты']['packing'].value_counts()

packing
1 кг          18
80 г           8
370 мл         8
300 г          6
1 л            6
2 кг           4
10 штук        4
5 л            3
10 шт          3
500 мл         3
260 мл         2
220 г          2
500 г          2
580 мл         2
50 г           2
850 мл         2
10 г           2
350 г          1
10 кг          1
450 г          1
2 г            1
425 мл         1
100 г          1
75 г           1
150 г          1
180 г          1
1,5 г          1
30 шт          1
0.5 л          1
200 мл         1
1 упаковка     1
2,2 кг         1
Name: count, dtype: int64

>Задача №1: получить столбец с ценой за 100г/100мл ингридиента
Для этого: 
* для упаковки "_ кг" привести к цене за 100 г 
* для упаковки "_ л" привести к цене за 100 мл
* для упаковки "_ штук" привести к цене за 1 штуку
* для упаковки "1 упаковка" оставить как есть
* для упаковки "_ кг" привести к цене за 100 г

In [40]:
def to_gramm(string):
    string = string.split(' ')

    if string[1] == 'кг':
        if ',' in string[0]:
            return float(string[0].replace(',','.')) * 1000
        else:
            return float(string[0]) * 1000
    elif string[1] == 'л':
            if ',' in string[0]:
                return float(string[0].replace(',','.')) * 1000
            else:
                return float(string[0]) * 1000
    else:
        if ',' in string[0]:
            return float(string[0].replace(',','.'))
        else:
            return float(string[0])

In [41]:
to_gramm('1 упаковка')

1.0

In [42]:
def units(string):
    string = string.split(' ')

    if string[1] == 'кг' or string[1] == 'г':
        return 'г'
    elif string[1] == 'л' or string[1] == 'мл':
        return 'мл'
    elif string[1] == 'штук' or string[1] == 'шт' or string[1] == 'лист':
        return 'шт'
    elif string[1] == 'упаковка':
        return 'уп'
    else:
        pass


In [43]:
data['units'] = data['packing'].apply(units)

In [44]:
data['package weight'] = data['packing'].apply(to_gramm)

In [45]:
data['price for unit'] = 0
data.head(3)

,id,name,packing,store,price,type,units,package weight,price for unit
0,1,сахар,1 кг,лента,62.99,продукты,г,1000.0,0
1,2,печенье на декор,1 упаковка,лента,89.99,декор,уп,1.0,0
2,3,сгущенка рогачев,370 мл,лента,164.99,продукты,мл,370.0,0


In [46]:
mask1 = (data['units'] == 'шт') | (data['units'] == 'уп')
mask2 = (data['units'] == 'г') | (data['units'] == 'мл')

In [47]:
data.loc[mask1,'price for unit'] = data.loc[mask1, 'price'] / data.loc[mask1,'package weight']
data.loc[mask2,'price for unit'] = (100 / data.loc[mask2, 'package weight']) * data.loc[mask2,'price']

In [48]:
data = data.drop('packing', axis = 1)

In [49]:
data

,id,name,store,price,type,units,package weight,price for unit
0,1,сахар,лента,62.99,продукты,г,1000.0,6.299000
1,2,печенье на декор,лента,89.99,декор,уп,1.0,89.990000
2,3,сгущенка рогачев,лента,164.99,продукты,мл,370.0,44.591892
3,4,малина и голубика в сердце,перекресток,579.99,декор,уп,1.0,579.990000
4,5,яйцо с1 окское,лента,109.99,продукты,шт,10.0,10.999000
...,...,...,...,...,...,...,...,...
133,134,глазурь темная,ольга,560.00,декор,г,1000.0,56.000000
134,135,трайфлы,ольга,1200.00,упаковка,шт,100.0,12.000000
135,136,коробка кап6,ольга,60.00,упаковка,шт,1.0,60.000000
136,137,яйцо с0 365 дней,лента,99.99,продукты,шт,10.0,9.999000


In [56]:
# Покажи все предложения по сахару и магазину по убыванию цены
data[data['type'] == 'продукты'].sort_values(by=['name','price for unit']).head(50)

,id,name,store,price,type,units,package weight,price for unit
33,34,ананасы кольцами лента 580 мл,лента,179.99,продукты,мл,580.0,31.032759
84,85,ананасы кусочками 365 дней,лента,149.99,продукты,мл,425.0,35.291765
90,91,ананасы кусочками лента,лента,169.99,продукты,мл,580.0,29.308621
63,64,арахис соленый,пятерочка,33.99,продукты,г,80.0,42.487500
8,9,арахис соленый 150 г,лента,49.99,продукты,г,150.0,33.326667
7,8,арахис соленый 50 г,лента,29.99,продукты,г,50.0,59.980000
85,86,ванилин dr.bakers интенсив,лента,13.99,продукты,г,2.0,699.500000
49,50,ванилин haas 1.5г,лента,6.99,продукты,г,1.5,466.000000
119,120,вишня с/м,ольга,580.00,продукты,г,1000.0,58.000000
71,72,вишня с/м,перекресток,399.99,продукты,г,300.0,133.330000


>Задача №2
* Необходимо создать таблицы с рецептами
* В таблице с ингридиентами определиться по какому критерию будем выбирать продукт среди одинаковых имен
* Соединить таблицу рецепт с таблицей ингридиент.